In [2]:
import cv2
import numpy as np
import tensorflow as tf


In [ ]:
# ===== Load model =====
model = tf.keras.models.load_model("mobilenetv2_96_best_model.h5")
label_names = ["NO_THREAT", "THREAT"]  # adjust if needed
img_size = (96, 96)

# ===== Open webcam =====
cap = cv2.VideoCapture(0)  # 0 for default camera

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to capture frame.")
        break

    # Preprocess frame
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, img_size)
    img_normalized = img_resized.astype(np.float32) / 255.0
    img_batch = np.expand_dims(img_normalized, axis=0)

    # Prediction
    preds = model.predict(img_batch, verbose=0)
    pred_class = np.argmax(preds, axis=1)[0]
    confidence = preds[0][pred_class]

    # Overlay prediction
    label_text = f"{label_names[pred_class]}: {confidence*100:.1f}%"
    color = (0, 255, 0) if pred_class == 0 else (0, 0, 255)  # green for NO_THREAT, red for THREAT
    cv2.putText(frame, label_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                1, color, 2, cv2.LINE_AA)

    # Display
    cv2.imshow("Real-time Threat Detection", frame)

    # Exit on 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
